# KV Checkpoint Accuracy on Local Hardware

This notebook evaluates synthetic KV checkpoints by checking whether the model predicts the correct value token after a query token.

Use `EXPERIMENT_PRESET` near the top to switch between the local 125M Orbax checkpoints and the Hugging Face `miki-aisle/thesis-kv14m-wide` checkpoint.

The metrics are split by whether the query has appeared earlier in the same synthetic document, and for repeated queries, whether the previous value token is still inside the 1024-token sliding window.

Run the cells from top to bottom. Defaults are intentionally small for a MacBook smoke run; increase `MAX_DOCS` after confirming the first pass works.

In [1]:
# Run this before importing JAX.
import os
import sys
from pathlib import Path

REPO_ROOT = Path("/Users/miki/aisle-thesis")
E2E_DIR = REPO_ROOT / "e2e"
os.chdir(E2E_DIR)

if str(E2E_DIR) not in sys.path:
    sys.path.insert(0, str(E2E_DIR))

os.environ.setdefault("JAX_PLATFORM_NAME", "cpu")
os.environ.setdefault("XLA_FLAGS", "--xla_force_host_platform_device_count=1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

print(f"cwd: {Path.cwd()}")
print(f"JAX_PLATFORM_NAME={os.environ.get('JAX_PLATFORM_NAME')}")
print(f"XLA_FLAGS={os.environ.get('XLA_FLAGS')}")

cwd: /Users/miki/aisle-thesis/e2e
JAX_PLATFORM_NAME=cpu
XLA_FLAGS=--xla_force_host_platform_device_count=1


In [2]:
from dataclasses import dataclass
import importlib
import sys

if sys.version_info < (3, 12):
    raise RuntimeError("This notebook needs Python 3.12+ because the e2e model code uses Python 3.12 syntax.")

required_imports = {
    "equinox": "equinox",
    "hydra": "hydra-core",
    "jax": "jax",
    "orbax.checkpoint": "orbax-checkpoint",
    "grain.python": "grain",
    "huggingface_hub": "huggingface-hub",
    "pandas": "pandas",
    "safetensors.flax": "safetensors[flax]",
}
missing_packages = []
for import_name, package_name in required_imports.items():
    try:
        importlib.import_module(import_name)
    except ModuleNotFoundError:
        missing_packages.append(package_name)

if missing_packages:
    raise ModuleNotFoundError(
        "Missing packages needed for checkpoint inference: "
        + ", ".join(sorted(set(missing_packages)))
        + ". Run this notebook with the e2e environment from e2e/pyproject.toml."
    )

import equinox as eqx
import hydra
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
from omegaconf import OmegaConf, open_dict
from tqdm.auto import tqdm

from ttt.config import register_configs
from ttt.dataloader.lm_dataset import SyntheticKVDataset
from ttt.infra.checkpoint import Checkpointer, unify_dict_with_eqx_module
from ttt.infra.hf_weights import load_hf_into_model
from ttt.model.data import Batch
from ttt.model.sharding import ModelSharding
from ttt.model.transformer import MetaModel
from ttt.utils.jax_utils import eval_shape_and_sharding, initialize_distibuted, set_random_seed

register_configs()
print(jax.devices())


@dataclass(frozen=True)
class CheckpointSpec:
    label: str
    experiment: str
    seq_len: int
    attention: str
    source: str = "orbax"  # "orbax" or "hf"
    resume_exp_name: str = ""
    exp_folder: str = "kv-pretrain"
    hf_repo_id: str = ""
    resume_step: int | None = None


# Switch this between "kv14m_wide_local_orbax" and "125m_local_orbax".
EXPERIMENT_PRESET = "kv14m_wide_local_orbax"

CHECKPOINT_ROOT = E2E_DIR / "checkpoints"
DEFAULT_RESUME_STEP = 14999
MAX_DOCS = 8
LOGIT_CHUNK_SIZE = 32
EVAL_SPLIT = "eval"
SLIDING_WINDOW_SIZE = 1024

# "full_document": evaluate each checkpoint on its configured sequence length.
# "local_1k_docs": evaluate checkpoints on 1K synthetic sequences, matching chunk-local SWA training.
EVAL_MODE = "full_document"
LOCAL_EVAL_SEQ_LEN = 1024

PRESETS = {
    "kv14m_wide_local_orbax": [
        CheckpointSpec(
            label="1K KV14M wide FA",
            experiment="kv14m_wide/kv_pretrain/pretrain-1K-kv14m-wide-fa-kv-10Ksteps",
            resume_exp_name="pretrain-1K-kv14m-wide-fa-kv-10Ksteps",
            exp_folder="kv14m-wide-kv-pretrain",
            seq_len=1024,
            attention="fa",
            resume_step=9999,
        ),
        CheckpointSpec(
            label="8K KV14M wide FA",
            experiment="kv14m_wide/kv_pretrain/pretrain-8K-kv14m-wide-fa-kv-10Ksteps",
            resume_exp_name="pretrain-8K-kv14m-wide-fa-kv-10Ksteps",
            exp_folder="kv14m-wide-kv-pretrain",
            seq_len=8192,
            attention="fa",
            resume_step=9999,
        ),
        CheckpointSpec(
            label="8K KV14M wide SWA-1K",
            experiment="kv14m_wide/kv_pretrain/pretrain-8K-kv14m-wide-swa1k-kv-10Ksteps",
            resume_exp_name="pretrain-8K-kv14m-wide-swa1k-kv-10Ksteps",
            exp_folder="kv14m-wide-kv-pretrain",
            seq_len=8192,
            attention="swa",
            resume_step=9999,
        ),
    ],
    "125m_local_orbax": [
        CheckpointSpec(
            label="2K FA",
            experiment="125m/kv_pretrain/pretrain-2K-125m-fa-kv-15Ksteps",
            resume_exp_name="pretrain-2K-125m-fa-kv-15Ksteps",
            seq_len=2048,
            attention="fa",
            resume_step=DEFAULT_RESUME_STEP,
        ),
        CheckpointSpec(
            label="4K FA",
            experiment="125m/kv_pretrain/pretrain-4K-125m-fa-kv-15Ksteps",
            resume_exp_name="pretrain-4K-125m-fa-kv-15Ksteps",
            seq_len=4096,
            attention="fa",
            resume_step=DEFAULT_RESUME_STEP,
        ),
        CheckpointSpec(
            label="2K SWA",
            experiment="125m/kv_pretrain/pretrain-2K-125m-swa-kv-15Ksteps",
            resume_exp_name="pretrain-2K-125m-swa-kv-15Ksteps",
            seq_len=2048,
            attention="swa",
            resume_step=DEFAULT_RESUME_STEP,
        ),
        CheckpointSpec(
            label="4K SWA",
            experiment="125m/kv_pretrain/pretrain-4K-125m-swa-kv-15Ksteps",
            resume_exp_name="pretrain-4K-125m-swa-kv-15Ksteps",
            seq_len=4096,
            attention="swa",
            resume_step=DEFAULT_RESUME_STEP,
        ),
    ],
}

CHECKPOINTS = PRESETS[EXPERIMENT_PRESET]
pd.DataFrame(CHECKPOINTS)

RuntimeError: This notebook needs Python 3.12+ because the e2e model code uses Python 3.12 syntax.

In [ ]:
# Optional: download local Orbax checkpoints from the Modal volume to CHECKPOINT_ROOT.
# HF presets download model.safetensors through huggingface_hub during load.
RUN_DOWNLOADS = False


def checkpoint_step_dir(spec: CheckpointSpec) -> Path | None:
    if spec.source != "orbax":
        return None
    step = spec.resume_step if spec.resume_step is not None else DEFAULT_RESUME_STEP
    return CHECKPOINT_ROOT / spec.exp_folder / spec.resume_exp_name / str(step)


def download_missing_checkpoints() -> None:
    import subprocess

    for spec in CHECKPOINTS:
        if spec.source != "orbax":
            print(f"skip download for {spec.label}: source={spec.source}")
            continue

        step_dir = checkpoint_step_dir(spec)
        if step_dir is not None and step_dir.exists():
            print(f"found {step_dir}")
            continue

        destination = CHECKPOINT_ROOT / spec.exp_folder
        destination.mkdir(parents=True, exist_ok=True)
        remote_path = f"/{spec.exp_folder}/{spec.resume_exp_name}"
        cmd = [
            "pixi",
            "run",
            "modal",
            "volume",
            "get",
            "e2e-checkpoints",
            remote_path,
            str(destination),
        ]
        print("+", " ".join(cmd))
        subprocess.run(cmd, cwd=REPO_ROOT, check=True)


if RUN_DOWNLOADS:
    download_missing_checkpoints()

missing = [spec for spec in CHECKPOINTS if spec.source == "orbax" and not checkpoint_step_dir(spec).exists()]
if missing:
    print("Missing local checkpoint step dirs:")
    for spec in missing:
        print(" -", checkpoint_step_dir(spec))
    print("Set RUN_DOWNLOADS=True above, or update CHECKPOINT_ROOT to an existing local copy.")
else:
    print("All local Orbax checkpoint step dirs found, or current preset uses HF weights.")

In [ ]:
def effective_seq_len(spec: CheckpointSpec) -> int:
    if EVAL_MODE == "local_1k_docs":
        return min(spec.seq_len, LOCAL_EVAL_SEQ_LEN)
    if EVAL_MODE == "full_document":
        return spec.seq_len
    raise ValueError(f"Unknown EVAL_MODE={EVAL_MODE!r}")


def compose_local_config(spec: CheckpointSpec):
    seq_len = effective_seq_len(spec)
    overrides = [
        "+deploy=interactive",
        f"+experiment={spec.experiment}",
        f"training.checkpoint_path={CHECKPOINT_ROOT}",
        "training.log_wandb=false",
        "backend.backend=cpu",
        "backend.distributed=false",
        "backend.num_devices=1",
        "backend.compilation_cache_dir=/tmp/jax_cache_kv_notebook",
        "training.n_data_parallel=1",
        "training.n_state_parallel=1",
        "model.force_flash=false",
    ]
    if spec.source == "orbax":
        step = spec.resume_step if spec.resume_step is not None else DEFAULT_RESUME_STEP
        overrides.extend(["training.load_part=params", f"training.resume_step={step}"])

    with hydra.initialize_config_dir(config_dir=str(E2E_DIR / "configs"), version_base=None):
        cfg = hydra.compose(config_name="config", overrides=overrides)

    with open_dict(cfg):
        cfg.training.resume_exp_name = spec.resume_exp_name
        cfg.training.seq_length = seq_len
        cfg.model.seq_len = seq_len
        cfg.training.eval_batch_size = 1
        cfg.training.shuffle_train = False
        cfg.backend.backend = "cpu"
        cfg.backend.distributed = False
        cfg.backend.num_devices = 1
        cfg.model.force_flash = False

    OmegaConf.resolve(cfg)
    return cfg


def load_checkpoint_model(spec: CheckpointSpec):
    cfg = compose_local_config(spec)

    initialize_distibuted(cfg.backend)
    key = set_random_seed(cfg.training.model_seed)
    model_sharding = ModelSharding(cfg)
    mesh = model_sharding.mesh

    @eqx.filter_jit
    def create_sharded_model_and_state():
        model, state = eqx.nn.make_with_state(MetaModel)(cfg, key=key)
        state = jax.device_put(state, jax.NamedSharding(mesh, jax.sharding.PartitionSpec()))
        model = model_sharding.shard_params(model)
        return model, state

    with mesh:
        model, state = create_sharded_model_and_state()

        if spec.source == "hf":
            print(f"Loading {spec.label} from HF repo {spec.hf_repo_id}")
            model = load_hf_into_model(spec.hf_repo_id, model, cfg.model)
        elif spec.source == "orbax":
            print(f"Loading {spec.label} from {cfg.checkpoint.resume_checkpoint_dir}:{cfg.training.resume_step}")
            abstract_model_weights = eval_shape_and_sharding(lambda: create_sharded_model_and_state()[0].weights())
            checkpointer = Checkpointer(config=cfg, for_saving=False)
            out_state = checkpointer.load_checkpoint(
                step=cfg.training.resume_step,
                targets={"model_weights": abstract_model_weights},
                restore=cfg.training.load_part,
            )
            model = unify_dict_with_eqx_module(out_state["model_weights"], model)[0]
            checkpointer.close()
        else:
            raise ValueError(f"Unknown checkpoint source: {spec.source!r}")

        state = state.set(model.step_index, jnp.array(jnp.iinfo(jnp.int32).max - 100, dtype=jnp.int32))

    return cfg, model, state, mesh

In [ ]:
@eqx.filter_jit
def hidden_states_for_tokens(model: MetaModel, state, input_ids, target_tokens, loss_masks):
    seq = Batch(input_ids=input_ids, target_tokens=target_tokens, loss_masks=loss_masks)
    return model.language_model.model(state, seq).last_hidden_state


@eqx.filter_jit
def predict_value_tokens(model: MetaModel, hidden_chunk):
    logits = model.language_model.wte_disembed_call(hidden_chunk)
    return jnp.argmax(logits, axis=-1).astype(jnp.int32)


def value_target_table(tokens: np.ndarray, loss_masks: np.ndarray, *, sliding_window_size: int) -> pd.DataFrame:
    rows = []
    last_query_pos: dict[int, int] = {}

    for pos in np.flatnonzero(loss_masks):
        query_token = int(tokens[pos])
        value_token = int(tokens[pos + 1])
        previous_query_pos = last_query_pos.get(query_token)

        if previous_query_pos is None:
            already_present = False
            within_window = False
            previous_value_distance = -1
            bucket = "not_previously_present"
        else:
            previous_value_pos = previous_query_pos + 1
            previous_value_distance = int(pos - previous_value_pos)
            already_present = True
            within_window = previous_value_distance < sliding_window_size
            bucket = "present_within_window" if within_window else "present_outside_window"

        rows.append(
            {
                "position": int(pos),
                "query_token": query_token,
                "value_token": value_token,
                "already_present": already_present,
                "within_window": within_window,
                "previous_value_distance": previous_value_distance,
                "bucket": bucket,
            }
        )
        last_query_pos[query_token] = int(pos)

    return pd.DataFrame(rows)


def evaluate_document(model: MetaModel, state, mesh, tokens: np.ndarray, loss_masks: np.ndarray) -> pd.DataFrame:
    target_df = value_target_table(tokens, loss_masks, sliding_window_size=SLIDING_WINDOW_SIZE)
    input_ids = jnp.asarray(tokens[:-1], dtype=jnp.int32)
    target_tokens = jnp.asarray(tokens[1:], dtype=jnp.int32)
    loss_masks_jax = jnp.asarray(loss_masks, dtype=bool)

    with mesh:
        hidden = hidden_states_for_tokens(model, state, input_ids, target_tokens, loss_masks_jax)

        predictions = []
        positions = target_df["position"].to_numpy(dtype=np.int32)
        for start in range(0, len(positions), LOGIT_CHUNK_SIZE):
            pos_chunk = jnp.asarray(positions[start : start + LOGIT_CHUNK_SIZE])
            hidden_chunk = hidden[pos_chunk]
            pred_chunk = predict_value_tokens(model, hidden_chunk)
            predictions.append(np.asarray(jax.device_get(pred_chunk)))

    predicted = np.concatenate(predictions) if predictions else np.array([], dtype=np.int32)
    target_df["predicted_value_token"] = predicted
    target_df["correct"] = target_df["predicted_value_token"].to_numpy() == target_df["value_token"].to_numpy()
    return target_df


def make_eval_dataset(cfg, spec: CheckpointSpec) -> SyntheticKVDataset:
    return SyntheticKVDataset(
        seq_len=effective_seq_len(spec),
        vocab_size=cfg.model.vocab_size,
        bos_token_id=cfg.model.bos_token_id,
        eos_token_id=cfg.model.eos_token_id,
        num_pairs=cfg.dataset.synthetic_num_pairs,
        num_docs=cfg.dataset.synthetic_num_docs,
        seed=cfg.dataset.synthetic_seed,
        data_partition=EVAL_SPLIT,
        eval_fraction=cfg.training.eval_fraction,
    )


def evaluate_checkpoint(spec: CheckpointSpec) -> pd.DataFrame:
    cfg, model, state, mesh = load_checkpoint_model(spec)
    dataset = make_eval_dataset(cfg, spec)
    n_docs = len(dataset) if MAX_DOCS == 0 else min(MAX_DOCS, len(dataset))

    doc_results = []
    for doc_idx in tqdm(range(n_docs), desc=f"eval {spec.label}"):
        tokens, loss_masks = dataset[doc_idx]
        doc_df = evaluate_document(model, state, mesh, np.asarray(tokens), np.asarray(loss_masks, dtype=bool))
        doc_df.insert(0, "doc_idx", doc_idx)
        doc_results.append(doc_df)

    result = pd.concat(doc_results, ignore_index=True)
    result.insert(0, "eval_mode", EVAL_MODE)
    result.insert(0, "attention", spec.attention)
    result.insert(0, "eval_seq_len", effective_seq_len(spec))
    result.insert(0, "train_seq_len", spec.seq_len)
    result.insert(0, "checkpoint", spec.label)

    del model, state
    jax.clear_caches()
    return result


def summarize_accuracy(rows: pd.DataFrame) -> pd.DataFrame:
    summary = (
        rows.groupby(["checkpoint", "attention", "train_seq_len", "eval_seq_len", "eval_mode", "bucket", "already_present", "within_window"], as_index=False)
        .agg(correct=("correct", "sum"), total=("correct", "size"))
        .sort_values(["eval_seq_len", "attention", "bucket"])
    )
    summary["accuracy"] = summary["correct"] / summary["total"]
    return summary

In [ ]:
missing = [spec for spec in CHECKPOINTS if spec.source == "orbax" and not checkpoint_step_dir(spec).exists()]
if missing:
    missing_paths = "\n".join(str(checkpoint_step_dir(spec)) for spec in missing)
    raise FileNotFoundError(
        "Missing checkpoint step directories. Run the download cell with RUN_DOWNLOADS=True or update CHECKPOINT_ROOT.\n"
        + missing_paths
    )

all_rows = []
for spec in CHECKPOINTS:
    all_rows.append(evaluate_checkpoint(spec))

raw_results = pd.concat(all_rows, ignore_index=True)
summary = summarize_accuracy(raw_results)
summary

In [ ]:
accuracy_pivot = summary.pivot_table(
    index=["checkpoint", "train_seq_len", "eval_seq_len", "eval_mode", "attention"],
    columns="bucket",
    values="accuracy",
)
accuracy_pivot

In [ ]:
import matplotlib.pyplot as plt

plot_df = summary.copy()
plot_df["series"] = plot_df["checkpoint"] + " / " + plot_df["bucket"]

ax = plot_df.plot.bar(x="series", y="accuracy", figsize=(12, 4), legend=False)
ax.set_ylim(0.0, 1.0)
ax.set_ylabel("value-token accuracy")
ax.set_xlabel("")
ax.set_title(f"Synthetic KV value-token accuracy over {MAX_DOCS if MAX_DOCS else 'all'} eval docs")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()